# Simulation and model smoke test

The fixed initial score `c0` is supplied explicitly to both the simulator and `model_coupling`.


In [ ]:
import arviz as az
import jax
import matplotlib.pyplot as plt
import numpy as np
import numpyro
import re
from pathlib import Path
from matplotlib.lines import Line2D
from coupling_sources.core.data import GainConstants, StimulusSpec
from coupling_sources.core.model import model_coupling
from coupling_sources.core.priors import CouplingPriors
from coupling_sources.core.simulator import simulate_coupling
from coupling_sources.core.stimuli import generate_stimulus

In [ ]:
numpyro.set_host_device_count(4)
numpyro.set_platform("cpu")
C0 = 0.0
GAINS = GainConstants(g_star_y=19.946, g_star_z=28.121)

In [ ]:
stimuli = generate_stimulus(
    StimulusSpec(
        seed=101,
        n_participants=1,
        n_sources=4,
        n_rounds=2,
        n_trials_full=4,
        n_trials_partial=4,
        sigma=0.55,
        lam=0.8,
        resultant_floor_full=0.6,
        resultant_floor_leave_one_out=0.6,
        coupled_sources=(0, 1),
    )
)

simulation = simulate_coupling(
    seed=201,
    batch=stimuli,
    parameters={"learningRate": 0.2, "gamma_y": 1.0, "gamma_z": 1.0, "vmConcentration": 8.0, "c0": C0},
    gains=GAINS,
)
simulation.stimuli.visible_reports.shape, simulation.responses.shape

In [ ]:
seeded_model = numpyro.handlers.seed(model_coupling, jax.random.PRNGKey(0))
seeded_model(simulation, C0, GAINS)
"model ran"

## Render the model with three trials

This section creates a minimal three-trial batch and renders the NumPyro model graph against it.


In [ ]:
render_stimuli = generate_stimulus(
    StimulusSpec(
        seed=301,
        n_participants=1,
        n_sources=4,
        n_rounds=1,
        n_trials_full=2,
        n_trials_partial=0,
        sigma=0.55,
        lam=0.8,
        resultant_floor_full=0.6,
        resultant_floor_leave_one_out=0.6,
        coupled_sources=(0, 1),
    )
)

render_simulation = simulate_coupling(
    seed=302,
    batch=render_stimuli,
    parameters={"learningRate": 0.2, "gamma_y": 1.0, "gamma_z": 1.0, "vmConcentration": 16.0, "c0": C0},
    gains=GAINS,
)

print("visible_reports shape:", render_simulation.stimuli.visible_reports.shape)
graph = numpyro.render_model(
    model_coupling,
    model_args=(render_simulation, C0, GAINS),
    render_distributions=False,
)
dot_source = graph.source
dot_source = re.sub(
    r'(label\s*=\s*"[^"]*)~\s*',
    r'\1',
    dot_source,
)
Path("model.dot").write_text(
    dot_source,
    encoding="utf-8",
)
graph

## Stimuli, simulation, MCMC fit, and posterior recovery

This section runs the full workflow explicitly: generate stimuli, simulate responses from known parameters, fit the model with MCMC, then overlay posterior distributions with the ground-truth parameter values.


In [ ]:
PARAMETER_SITES = CouplingPriors.parameter_sites()
GROUND_TRUTH = {"learningRate": 0.436240, "gamma_y": 15.597406 / GAINS.g_star_y, "gamma_z": 6.165416 / GAINS.g_star_z, "vmConcentration": 2.590899, "c0": C0}

fit_stimuli = generate_stimulus(
    StimulusSpec(
        seed=401,
        n_participants=1,
        n_sources=4,
        n_rounds=1,
        n_trials_full=20,
        n_trials_partial=20,
        sigma=0.55,
        lam=0.8,
        resultant_floor_full=0.6,
        resultant_floor_leave_one_out=0.6,
        coupled_sources=(0, 1),
    )
)

fit_stimuli.visible_reports.shape

In [ ]:
fit_simulation = simulate_coupling(
    seed=402,
    batch=fit_stimuli,
    parameters=GROUND_TRUTH,
    gains=GAINS,
)

fit_simulation.responses.shape

In [ ]:
kernel = numpyro.infer.NUTS(
    model_coupling,
    target_accept_prob=0.8,
    max_tree_depth=8,
)
mcmc = numpyro.infer.MCMC(
    kernel,
    num_warmup=200,
    num_samples=1000,
    num_chains=4,
    progress_bar=True,
)
mcmc.run(
    jax.random.PRNGKey(403),
    fit_simulation,
    GROUND_TRUTH["c0"],
    GAINS,
    extra_fields=("diverging",),
)

posterior_samples = {
    name: np.asarray(values).reshape(-1)
    for name, values in mcmc.get_samples().items()
    if name in PARAMETER_SITES
}
mcmc.print_summary()

In [ ]:
prior_samples = CouplingPriors.sample_parameter_draws(seed=404, draws=5_000, c0=C0)
idata = az.from_dict(
    {"posterior": posterior_samples, "prior": prior_samples},
    sample_dims=["draw"],
)

pair_args = dict(
    var_names=list(PARAMETER_SITES),
    triangle="lower",
    marginal=True,
    marginal_kind="kde",
    backend="matplotlib",
)
styles = {
    "posterior": {
        "scatter": {"color": "tab:blue", "alpha": 0.35, "s": 4},
        "dist": {"color": "tab:blue"},
    },
    "prior": {
        "scatter": {"color": "0.35", "alpha": 0.08, "s": 4},
        "dist": {"color": "0.35", "linestyle": "--"},
    },
}

pair_plot = az.plot_pair(
    idata,
    group="posterior",
    visuals=styles["posterior"],
    figure_kwargs={"figsize": (12, 12)},
    **pair_args,
)
az.plot_pair(
    idata,
    group="prior",
    plot_matrix=pair_plot,
    visuals=styles["prior"],
    **pair_args,
)

axes = pair_plot.viz["plot"].values
for row, y_parameter in enumerate(PARAMETER_SITES):
    for col, x_parameter in enumerate(PARAMETER_SITES[: row + 1]):
        ax = axes[row, col]
        if row == col:
            values = posterior_samples[x_parameter]
            ax.axvline(GROUND_TRUTH[x_parameter], color="black", linestyle="--", linewidth=1.0)
            lo, mid, hi = np.quantile(values, [0.05, 0.5, 0.95])
            ax.set_title(f"{x_parameter} = {mid:.2f} [{lo:.2f}, {hi:.2f}]", fontsize=9)
            ax.set_yticks([])
            ax.set_ylabel("")
        else:
            ax.axvline(GROUND_TRUTH[x_parameter], color="black", linestyle="--", linewidth=0.8, alpha=0.65)
            ax.axhline(GROUND_TRUTH[y_parameter], color="black", linestyle="--", linewidth=0.8, alpha=0.65)
            ax.plot(GROUND_TRUTH[x_parameter], GROUND_TRUTH[y_parameter], "k+", markersize=5)

axes[0, 0].legend(
    handles=[
        Line2D([], [], color="tab:blue", label="posterior"),
        Line2D([], [], color="0.35", linestyle="--", label="prior"),
        Line2D([], [], color="black", linestyle="--", label="ground truth"),
    ],
    frameon=False,
    fontsize=8,
)
plt.gcf().suptitle("Pairwise prior and posterior distributions", y=1.0)
plt.gcf().tight_layout()
plt.show()